# Financial Performance Forecasting using Machine Learning

This notebook studies whether historical financial indicators can be used to forecast subsequent-year financial performance for firms/banks.

**Dataset:** 213 firms/banks with seven yearly blocks of financial indicators.

**Models:** Linear Regression and Support Vector Regression (linear kernel).

**Metric:** Root Mean Squared Error (RMSE).

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.svm import SVR
from sklearn.metrics import mean_squared_error


## 1. Load the dataset

The repository stores the dataset in `data/7_year.csv`. A Colab fallback is included so the notebook can also be used with the original Colab workflow.

In [ ]:
DATA_PATH = Path('data/7_year.csv')
if not DATA_PATH.exists():
    DATA_PATH = Path('/content/7_year.csv')

df = pd.read_csv(DATA_PATH)
print(f'Dataset shape: {df.shape}')
display(df.head())

## 2. Organize the yearly financial indicators

Each year contains the same seven financial indicators. The first six yearly blocks are used as historical feature sets and the final block provides the held-out target.

The target used by the original implementation is **Income from financial services**.

In [ ]:
features_per_year = [
    'Other income',
    'Profit after tax',
    'Total capital',
    'Reserves and funds',
    'Deposits (accepted by commercial banks)',
    'Current liabilities & provisions',
    'Income from financial services'
]

feature_blocks = []
target_blocks = []

for year in range(7):
    suffix = '' if year == 0 else f'.{year}'
    cols = [f + suffix for f in features_per_year]
    block = df[cols].copy()
    if year < 6:
        feature_blocks.append(block)
    target_blocks.append(block['Income from financial services' + suffix])

print(f'Historical feature years: {len(feature_blocks)}')
print(f'Rows: {len(df)}')

## 3. Year-ahead forecasting setup

For each historical step, the model learns from one year's seven indicators and predicts the next year's target. The final model is trained on the latest available historical transition and evaluated on the final held-out target.

In [ ]:
X_years = feature_blocks
y_years = target_blocks[:6]

def evaluate_model(model):
    yearly_rmse = []
    for i in range(len(X_years) - 1):
        X_train = X_years[i]
        y_train = y_years[i + 1]
        X_test = X_years[i + 1]
        y_test = y_years[i + 2] if i + 2 < len(y_years) else target_blocks[6]
        model.fit(X_train, y_train)
        pred = model.predict(X_test)
        yearly_rmse.append(np.sqrt(mean_squared_error(y_test, pred)))
    return yearly_rmse


## 4. Compare regression models

The current project compares Linear Regression with a linear-kernel Support Vector Regressor.

In [ ]:
models = {
    'Linear Regression': LinearRegression(),
    'Linear SVR': SVR(kernel='linear')
}

results = {}
for name, model in models.items():
    rmse_values = evaluate_model(model)
    results[name] = rmse_values

summary = pd.DataFrame({
    name: values for name, values in results.items()
}, index=[f'Step {i + 1}' for i in range(len(next(iter(results.values()))))])
summary.loc['Mean RMSE'] = summary.mean()
display(summary)

## 5. Final held-out evaluation

The final evaluation follows the original project logic: train on the latest historical transition and predict the final year's target.

In [ ]:
X_train_final = X_years[-2]
y_train_final = y_years[-1]
X_test_final = X_years[-1]
y_test_final = target_blocks[6]

final_scores = {}
for name, model in models.items():
    model.fit(X_train_final, y_train_final)
    prediction = model.predict(X_test_final)
    final_scores[name] = np.sqrt(mean_squared_error(y_test_final, prediction))

final_results = pd.Series(final_scores, name='RMSE').sort_values()
display(final_results.to_frame())

In [ ]:
plt.figure(figsize=(6, 4))
final_results.plot(kind='bar')
plt.ylabel('RMSE')
plt.xlabel('Model')
plt.title('Final Forecasting Error Comparison')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

## 6. Interpretation

A lower RMSE indicates lower prediction error for the evaluated target. The results should be interpreted as an experimental comparison on this dataset rather than as a guarantee of future financial performance.

### Future improvements
- Walk-forward / rolling-window validation
- Additional regression and ensemble models
- MAE and R² alongside RMSE
- Feature-importance analysis
- Missing-value and outlier analysis
- Evaluation across multiple forecast horizons